In [1]:
# ================================
# UAV CG & Fuselage Design Tool
# ================================

# -------------------------------
# 1. INPUT PARAMETERS
# -------------------------------

# Wing geometry (mm)
c_root = 416
c_tip = 163

# Wing position
x_LE = 400  # mm from nose

# Tail
tail_arm = 1000  # mm from wing LE

# Masses (kg)
payload = 5.0
avionics = 1.5
structure = 5.98
propulsion = 1.1
battery = 4.0

# Positions (mm)
x_payload = 400
x_avionics = 450
x_structure = 490
x_propulsion = x_LE + tail_arm  # tail position


# -------------------------------
# 2. MAC CALCULATION
# -------------------------------
lambda_ = c_tip / c_root

MAC = (2/3) * c_root * (1 + lambda_ + lambda_**2) / (1 + lambda_)

print(f"MAC = {MAC:.2f} mm")


# -------------------------------
# 3. TARGET CG
# -------------------------------
cg_rel = 0.30 * MAC  # 30% MAC
x_CG_target = x_LE + cg_rel

print(f"Target CG = {x_CG_target:.2f} mm from nose")


# -------------------------------
# 4. FIXED MOMENTS (no battery)
# -------------------------------
M_fixed = (
    payload * x_payload +
    avionics * x_avionics +
    structure * x_structure +
    propulsion * x_propulsion
)

mass_fixed = payload + avionics + structure + propulsion

print(f"Fixed Moment = {M_fixed:.2f}")
print(f"Fixed Mass = {mass_fixed:.2f} kg")


# -------------------------------
# 5. SOLVE BATTERY POSITION
# -------------------------------
total_mass = mass_fixed + battery

# Solve:
# x_CG_target = (M_fixed + battery * x_b) / total_mass

x_b = (x_CG_target * total_mass - M_fixed) / battery

print(f"Required Battery Position = {x_b:.2f} mm from nose")


# -------------------------------
# 6. CG WITH PAYLOAD
# -------------------------------
M_total = M_fixed + battery * x_b
x_CG_with_payload = M_total / total_mass

print(f"CG (with payload) = {x_CG_with_payload:.2f} mm")


# -------------------------------
# 7. CG WITHOUT PAYLOAD
# -------------------------------
M_without_payload = M_total - payload * x_payload
mass_without_payload = total_mass - payload

x_CG_without_payload = M_without_payload / mass_without_payload

print(f"CG (without payload) = {x_CG_without_payload:.2f} mm")


# -------------------------------
# 8. CG RANGE CHECK
# -------------------------------
cg_min = x_LE + 0.25 * MAC
cg_max = x_LE + 0.35 * MAC

print(f"\nSafe CG Range: {cg_min:.2f} mm to {cg_max:.2f} mm")

def check_cg(x):
    if x < cg_min:
        return "Too Forward"
    elif x > cg_max:
        return "Too Aft"
    else:
        return "Safe"

print(f"With Payload CG Status: {check_cg(x_CG_with_payload)}")
print(f"Without Payload CG Status: {check_cg(x_CG_without_payload)}")


# -------------------------------
# 9. OPTIONAL: BATTERY SWEEP
# -------------------------------
print("\nBattery Sweep Analysis:")

for xb_test in range(250, 500, 50):
    M_test = M_fixed + battery * xb_test
    CG_test = M_test / total_mass
    print(f"Battery @ {xb_test} mm -> CG = {CG_test:.2f} mm")

MAC = 307.93 mm
Target CG = 492.38 mm from nose
Fixed Moment = 7145.20
Fixed Mass = 13.58 kg
Required Battery Position = 377.70 mm from nose
CG (with payload) = 492.38 mm
CG (without payload) = 529.09 mm

Safe CG Range: 476.98 mm to 507.77 mm
With Payload CG Status: Safe
Without Payload CG Status: Too Aft

Battery Sweep Analysis:
Battery @ 250 mm -> CG = 463.32 mm
Battery @ 300 mm -> CG = 474.70 mm
Battery @ 350 mm -> CG = 486.08 mm
Battery @ 400 mm -> CG = 497.45 mm
Battery @ 450 mm -> CG = 508.83 mm
